In [2]:
import jax 
import jax.numpy as jnp
import haiku as hk
import optax

from probjax.nn.transformers import Transformer
from probjax.nn.tokenizer import scalarize, ScalarTokenizer
from probjax.nn.helpers import GaussianFourierEmbedding
from probjax.nn.loss_fn import denoising_score_matching_loss

from functools import partial

In [3]:

def gaussian_simulator(key, n):
    theta = jax.random.normal(key, (n, 2))
    x = theta +  1e-1 * jax.random.normal(key, theta.shape) + 2.
    return jnp.concatenate([theta,x], axis=-1)

data = gaussian_simulator(jax.random.PRNGKey(0), 10000).reshape(-1, 4, 1)
node_id = jnp.arange(data.shape[-2])[None].repeat(data.shape[0], axis=0)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [4]:

def model(t,data, data_id):
    tokenizer = ScalarTokenizer(4, len(node_id))
    time_embeder = GaussianFourierEmbedding(4, learnable=True)
    
    # Embedding
    tokens = tokenizer(data_id,data)
    time = time_embeder(t[..., None])

    model = Transformer(num_heads=4, num_layers=2, attn_size=4, widening_factor=10)
    h = model(tokens, context=time)
    out = hk.Linear(1)(h)
    return out

init_fn, model_fn = hk.without_apply_rng(hk.transform(model))
params = init_fn(jax.random.PRNGKey(0), jnp.zeros(data.shape[0],),data, node_id)

In [5]:
from probjax.distributions.sde import VPSDE
from probjax.distributions.discrete import Empirical

In [6]:
# VPSDE 
T = 1.
T_min = 1e-6
beta_min = 0.1
beta_max = 10.

# Defines the SDE
def beta(t):
    return beta_min + t * (beta_max - beta_min)

def drift_fn(t, x):
    return -0.5 * (beta(t)) * x

def diffusion_fn(t, x):
    # This migh requrire a square root
    return jnp.sqrt(beta(t)) * jnp.ones_like(x)

# Defines the marginal moments conditioned on x0
def marginal_mean(t, x0):
    phi = jnp.exp(-0.25 * t**2 * (beta_max - beta_min) - 0.5*t * beta_min)
    phi = phi.reshape(-1,1,1)
    return phi * x0

def marginal_std(t, x0):
    phi = jnp.exp(-0.5 * t**2 * (beta_max - beta_min) - t * beta_min)
    phi = phi.reshape(-1,1,1)
    return jnp.sqrt(1 - 1 * phi)

def weight_fn(t):
    t = t.reshape(-1,1,1)
    return 1-jnp.exp(-0.5 * (beta_max - beta_min) * t**2 - beta_min * t) + 1e-3

In [7]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

In [8]:
def loss_fn(params, key, data, node_id, mask = None):
    key_times, key_loss = jax.random.split(key)
    times = jax.random.uniform(key_times, (data.shape[0],))
    loss = denoising_score_matching_loss(params, key_loss, times, data, mask, node_id, model_fn = model_fn, mean_fn = marginal_mean, std_fn=marginal_std, weight_fn=weight_fn)
    return loss

@partial(jax.pmap, axis_name="num_devices")
def update(params, opt_state, key, data, node_id):
    loss, grads = jax.value_and_grad(loss_fn)(params, key, data, node_id)

    loss = jax.lax.pmean(loss, axis_name="num_devices")
    grads = jax.lax.pmean(grads, axis_name="num_devices")
    
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state
    

In [9]:
num_epochs = 50 
batch_size = 1000
num_devices = jax.device_count()
batch_size_per_device = batch_size // num_devices
num_steps = data.shape[0] // batch_size_per_device + 1

replicated_params = jax.tree_map(lambda x: jnp.array([x] * num_devices), params)
replicated_opt_state = jax.tree_map(lambda x: jnp.array([x] * num_devices), opt_state)

In [10]:
key = jax.random.PRNGKey(9)
for _ in range(num_epochs):
    l = 0
    for i in range(num_steps):
        key_batch, key_update = jax.random.split(key)
        data_batch = jax.random.choice(key_batch,data, shape=(num_devices, batch_size_per_device, ), axis=0)
        id_batch = jax.random.choice(key_batch,node_id, shape=(num_devices, batch_size_per_device, ), axis=0)
        loss, replicated_params, replicated_opt_state = update(replicated_params, replicated_opt_state, jax.random.split(key_update, (num_devices,)), data_batch, id_batch)
        l += loss[0] /num_steps

params = jax.tree_map(lambda x: x[0], replicated_params)

In [ ]:
jax